# Concentrated Fed-Batch Operating Mode Test Notebook

This notebook tests the **CONCENTRATED_FEDBATCH** operating mode implementation.

**Test Coverage:**
- Simulation setup and execution
- Single batch run with initial conditions
- Multi-batch run with auto-extracted initial conditions
- Data integrity validation
- Performance metrics
- Concentrated fed-batch-specific validations (mode transition, combined feed/perfusion)

## 1. Setup and Imports

In [1]:
import sys
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy
from animakinetica.models.gas_controllers import GasController

print("✓ Imports successful")

✓ Imports successful


## 2. Load Parameters and Initialize Simulation

In [2]:
# Load base parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "excess_kinetics_adjusted", "fed_batch", "perfusion"])

# Initialize simulation with CONCENTRATED_FEDBATCH mode
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.EXCESS,
    operating_mode=OperatingMode.CONCENTRATED_FEDBATCH,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    perfusion_strategy=PerfusionStrategy.CSPR,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False,  # Store detailed data for testing
    solver_method="BDF"
)

print(f"✓ Simulation initialized")
print(f"  Operating Mode: {sim.operating_mode}")
print(f"  Kinetics Model: {sim.kinetics}")
print(f"  Feed Strategy: {sim.feed_strategy}")
print(f"  Perfusion Strategy: {sim.perfusion_strategy}")
print(f"  Gas Controller: {sim.gas_controller}")

Loading parameters from 7 file(s) sequentially...
  [1/7] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/7] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/7] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/7] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/7] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\excess_kinetics\excess_kinetics_adjusted.yaml
  [6/7] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakin

## 4. Multi-Batch Run

In [3]:
# Run 5 repeated batches
num_batches = 5
multi_result = sim.run_simulation(number_of_batches=num_batches)

print(f"✓ {num_batches} batches completed")
print(f"\nSummary Statistics:")
print(f"  Average batch duration: {multi_result.projected_totals_stats['total_simulation_time_h']/num_batches:.2f} h")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")

print(f"\nBatch-by-Batch Results:")
for i, batch in enumerate(sim.data.batches, 1):
    print(f"  Batch {i}: {batch.t_end:.2f} h, Xv={batch.Xv_final:.2e} cells/L, V={batch.V_final:.2f} L")

run_simulation execution time: 625.696 ms
✓ 5 batches completed

Summary Statistics:
  Average batch duration: 192.00 h
  STY: 0.757 kg/m3/day

Batch-by-Batch Results:
  Batch 1: 216.48 h, Xv=5.00e+10 cells/L, V=2000.00 L
  Batch 2: 216.03 h, Xv=5.00e+10 cells/L, V=2000.00 L
  Batch 3: 216.03 h, Xv=5.00e+10 cells/L, V=2000.00 L
  Batch 4: 216.03 h, Xv=5.00e+10 cells/L, V=2000.00 L
  Batch 5: 95.43 h, Xv=9.02e+09 cells/L, V=600.00 L


## 5. Data Integrity Validation

In [4]:
# Test batch 1 data integrity
batch_1 = sim.data.batches[0]

print("Data Integrity Checks for Batch 1:")
print("\n1. State Variables:")
# y_points shape: (n_states, n_timepoints) - states are rows
# [Xv, C_Glc, C_Gln, C_Lac, C_Amm, V, C_O2, C_CT, C_H, C_Na, C_AA]
state_vars = ['Xv', 'C_Glc', 'C_Gln', 'C_Lac', 'C_Amm', 'V', 'C_O2', 'C_CT', 'C_H', 'C_Na', 'C_AA']
for idx, var in enumerate(state_vars):
    data = np.asarray(batch_1.y_points[idx], dtype=float)  # Ensure numeric array
    print(f"  {var}: {len(data)} points, range [{data.min():.2e}, {data.max():.2e}]")
    assert len(data) > 0, f"{var} data is empty"
    assert not np.any(np.isnan(data)), f"{var} contains NaN values"

print("\n2. Kinetic Rates:")
kinetic_rates = ['mu', 'q_Glc', 'q_Gln', 'q_Lac', 'q_Amm', 'q_AA', 'q_O2', 'q_CO2']
for rate in kinetic_rates:
    rate_list = getattr(batch_1, f'{rate}_list')
    print(f"  {rate}: {len(rate_list)} points, range [{rate_list.min():.2e}, {rate_list.max():.2e}]")
    assert len(rate_list) > 0, f"{rate}_list is empty"
    assert not np.any(np.isnan(rate_list)), f"{rate}_list contains NaN values"

print("\n3. Nutrient Factors:")
nf_vars = ['NF_glucose', 'NF_glutamine', 'NF_O2', 'NF']
for nf in nf_vars:
    nf_list = getattr(batch_1, f'{nf}_list')
    print(f"  {nf}: {len(nf_list)} points, range [{nf_list.min():.2e}, {nf_list.max():.2e}]")
    assert len(nf_list) > 0, f"{nf}_list is empty"
    assert np.all((nf_list >= 0) & (nf_list <= 1)), f"{nf}_list contains values outside [0,1]"

print("\n4. Inhibition Factors:")
if_vars = ['IF_O2', 'IF_CO2', 'IF_lactate', 'IF_ammonia', 'IF_osmolality', 'IF']
for ifvar in if_vars:
    if_list = getattr(batch_1, f'{ifvar}_list')
    print(f"  {ifvar}: {len(if_list)} points, range [{if_list.min():.2e}, {if_list.max():.2e}]")
    assert len(if_list) > 0, f"{ifvar}_list is empty"
    assert np.all((if_list >= 0) & (if_list <= 1)), f"{ifvar}_list contains values outside [0,1]"

print("\n5. Gas Control Variables:")
gas_vars = ['N', 'Fg', 'y_O2', 'kLa', 'OTR', 'CTR', 'C_CO2']
for gv in gas_vars:
    gv_list = getattr(batch_1, f'{gv}_list')
    print(f"  {gv}: {len(gv_list)} points, range [{gv_list.min():.2e}, {gv_list.max():.2e}]")
    assert len(gv_list) > 0, f"{gv}_list is empty"
    assert not np.any(np.isnan(gv_list)), f"{gv}_list contains NaN values"

print("\n6. pH Control Variables:")
print(f"  D_base: {len(batch_1.D_base_list)} points, range [{batch_1.D_base_list.min():.2e}, {batch_1.D_base_list.max():.2e}]")
assert len(batch_1.D_base_list) > 0, "D_base_list is empty"

print("\n7. Dilution Rate (Concentrated Fed-Batch-specific):")
D_list = batch_1.D_list
print(f"  D: {len(D_list)} points, range [{D_list.min():.4f}, {D_list.max():.4f}] 1/h")
assert len(D_list) > 0, "D_list is empty"
assert not np.any(np.isnan(D_list)), "D_list contains NaN values"

print("\n✓ All data integrity checks passed")


Data Integrity Checks for Batch 1:

1. State Variables:
  Xv: 284 points, range [6.00e+08, 5.00e+10]
  C_Glc: 284 points, range [7.98e-01, 2.50e+01]
  C_Gln: 284 points, range [2.41e+00, 1.00e+01]
  C_Lac: 284 points, range [0.00e+00, 9.05e+00]
  C_Amm: 284 points, range [0.00e+00, 2.77e+00]
  V: 284 points, range [6.00e+02, 2.00e+03]
  C_O2: 284 points, range [1.00e-02, 1.00e-02]
  C_CT: 284 points, range [1.87e+01, 2.90e+01]
  C_H: 284 points, range [6.31e-08, 6.31e-08]
  C_Na: 284 points, range [0.00e+00, 0.00e+00]
  C_AA: 284 points, range [4.24e-01, 7.67e+00]

2. Kinetic Rates:
  mu: 284 points, range [1.74e-02, 2.87e-02]
  q_Glc: 284 points, range [5.69e-11, 6.29e-11]
  q_Gln: 284 points, range [1.05e-11, 1.36e-11]
  q_Lac: 284 points, range [3.12e-12, 3.41e-11]
  q_Amm: 284 points, range [4.53e-12, 8.92e-12]
  q_AA: 284 points, range [8.87e-12, 1.46e-11]
  q_O2: 284 points, range [2.00e-10, 2.97e-10]
  q_CO2: 284 points, range [2.15e-10, 3.07e-10]

3. Nutrient Factors:
  NF_gluc

## 6. Performance Metrics

In [5]:
print("Performance Metrics (5-Batches):")
print(f"\n Total Biomass: {multi_result.projected_totals_stats['total_DCM_out_kg']:.2f} kg")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")
print(f"  Medium Efficiency: {multi_result.summary_stats['medium_use_efficiency_kg_m3']:.3f} kg/m3")
print(f"  Protein Use Efficiency: {multi_result.summary_stats['protein_use_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Nutrient Use Efficiency: {multi_result.summary_stats['nutrient_use_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Total Simulation Time: {multi_result.projected_totals_stats['total_simulation_time_h']:.2f} h")
print(f"  Number of Batches: {multi_result.projected_totals_stats['number_of_batches']}")


# Validate metrics are reasonable
assert multi_result.projected_totals_stats['total_DCM_out_kg'] > 0, "Total DCM should be positive"
assert multi_result.summary_stats['STY_kg_m3_day'] > 0, "STY should be positive"
assert 0 < multi_result.summary_stats['medium_use_efficiency_kg_m3'] < 1000, "Medium use efficiency should be between 0 and 100 kg/m3"
assert 0 < multi_result.summary_stats['protein_use_efficiency_kg_kg'] < 1, "Protein use efficiency should be between 0 and 1"
assert 0 < multi_result.summary_stats['nutrient_use_efficiency_kg_kg'] < 1, "Nutrient use efficiency should be between 0 and 1"
assert multi_result.projected_totals_stats['total_simulation_time_h'] > 0, "Total simulation time should be positive"
assert multi_result.projected_totals_stats['number_of_batches'] == num_batches, f"Number of batches should be {num_batches}"

print("\n✓ Performance metrics validated")

Performance Metrics (5-Batches):

 Total Biomass: 63.68 kg
  STY: 0.757 kg/m3/day
  Medium Efficiency: 0.738 kg/m3
  Protein Use Efficiency: 0.240 kg/kg
  Nutrient Use Efficiency: 0.133 kg/kg
  Total Simulation Time: 960.00 h
  Number of Batches: 5

✓ Performance metrics validated


## 7. Concentrated Fed-Batch-Specific Validations

In [6]:
print("Concentrated Fed-Batch-Specific Validations:")

# 1. Mode transition detection (fed-batch → perfusion)
print("\n1. Mode Transition Detection (Batch 1):")
D_list = batch_1.D_list
D_base_perfusion = float(params.perfusion.D_baseline)

# Identify transition point (when D starts increasing from 0 to perfusion)
# Fed-batch: D from feeding only (low)
# Perfusion: D from feeding + harvest (higher)
fed_batch_phase = D_list < (D_base_perfusion * 0.5)  # Heuristic threshold
perfusion_phase = D_list >= (D_base_perfusion * 0.5)

if np.any(perfusion_phase):
    transition_idx = np.argmax(perfusion_phase)
    t_transition = batch_1.t_points[transition_idx]
    Xv_at_transition = batch_1.y_points[0, transition_idx]
    V_at_transition = batch_1.y_points[5, transition_idx]
    C_Lac_at_transition = batch_1.y_points[3, transition_idx]
    C_Amm_at_transition = batch_1.y_points[4, transition_idx]
    D_at_transition = D_list[transition_idx]
    
    print(f"  Mode transition detected at t={t_transition:.2f} h")
    print(f"  Conditions at transition:")
    print(f"    Xv: {Xv_at_transition:.2e} cells/L")
    print(f"    Volume: {V_at_transition:.2f} L")
    print(f"    Lactate: {C_Lac_at_transition:.2f} mmol/L")
    print(f"    Ammonia: {C_Amm_at_transition:.2f} mmol/L")
    print(f"    Dilution rate: {D_at_transition:.4f} 1/h")
    
    # Check what triggered perfusion
    C_Lac_trig = float(params.perfusion.C_Lac_pf_trig) if params.perfusion.C_Lac_pf_trig is not None else float('inf')
    C_Amm_trig = float(params.perfusion.C_Amm_pf_trig) if params.perfusion.C_Amm_pf_trig is not None else float('inf')
    Xv_trig = float(params.perfusion.Xv_pf_trig) if params.perfusion.Xv_pf_trig is not None else float('inf')
    
    if C_Lac_at_transition >= C_Lac_trig:
        print(f"    → Triggered by lactate (threshold: {C_Lac_trig:.2f} mmol/L)")
    if C_Amm_at_transition >= C_Amm_trig:
        print(f"    → Triggered by ammonia (threshold: {C_Amm_trig:.2f} mmol/L)")
    if Xv_at_transition >= Xv_trig:
        print(f"    → Triggered by cell density (threshold: {Xv_trig:.2e} cells/L)")
else:
    print(f"  No perfusion transition detected (remained in fed-batch mode)")
    t_transition = None
print(f" Termination reason: {batch_1.termination_reason}")


# 2. Volume profile (increase during fed-batch, controlled during perfusion)
print("\n2. Volume Profile:")
V_initial = batch_1.y_points[5, 0]
V_final = batch_1.y_points[5, -1]
V_max = batch_1.y_points[5, :].max()
print(f"  V_initial: {V_initial:.2f} L")
print(f"  V_max: {V_max:.2f} L")
print(f"  V_final: {V_final:.2f} L")

if t_transition is not None:
    # Analyze volume before and after transition
    V_before = batch_1.y_points[5, :transition_idx]
    V_after = batch_1.y_points[5, transition_idx:]
    
    # Fed-batch phase: volume should increase
    V_increase_fedbatch = V_before[-1] - V_before[0] if len(V_before) > 0 else 0
    print(f"\n  Fed-batch phase (0 to {t_transition:.1f} h):")
    print(f"    Volume increase: {V_increase_fedbatch:.2f} L")
    
    # Perfusion phase: volume should be more controlled
    if len(V_after) > 1:
        V_std_perfusion = V_after.std()
        V_mean_perfusion = V_after.mean()
        print(f"  Perfusion phase ({t_transition:.1f} to {batch_1.t_end:.1f} h):")
        print(f"    Volume CV: {V_std_perfusion/V_mean_perfusion*100:.2f}%")

# 3. Dilution rate profile evolution
print("\n3. Dilution Rate Profile:")
D_initial = D_list[0]
D_final = D_list[-1]
D_max = D_list.max()
D_mean = D_list.mean()
print(f"  D_initial: {D_initial:.4f} 1/h")
print(f"  D_max: {D_max:.4f} 1/h")
print(f"  D_final: {D_final:.4f} 1/h")
print(f"  D_mean: {D_mean:.4f} 1/h")

# 4. Cell density achievement
print("\n4. Cell Density Profile:")
for i, batch in enumerate(sim.data.batches, 1):
    Xv_initial = batch.y_points[0, 0]
    Xv_max = batch.y_points[0, :].max()
    Xv_final = batch.y_points[0, -1]
    fold_increase = Xv_max / Xv_initial
    print(f"  Batch {i}:")
    print(f"    Xv_initial: {Xv_initial:.2e} cells/L")
    print(f"    Xv_max: {Xv_max:.2e} cells/L (fold: {fold_increase:.2f}x)")
    print(f"    Xv_final: {Xv_final:.2e} cells/L")
    assert Xv_max > Xv_initial, f"Batch {i}: No cell growth detected"

# 5. Metabolite control effectiveness
print("\n5. Metabolite Control:")
for i, batch in enumerate(sim.data.batches, 1):
    C_Lac_max = batch.y_points[3, :].max()
    C_Amm_max = batch.y_points[4, :].max()
    C_Lac_final = batch.y_points[3, -1]
    C_Amm_final = batch.y_points[4, -1]
    print(f"  Batch {i}:")
    print(f"    Lactate: max={C_Lac_max:.2f}, final={C_Lac_final:.2f} mmol/L")
    print(f"    Ammonia: max={C_Amm_max:.2f}, final={C_Amm_final:.2f} mmol/L")

# 6. Comparison with pure fed-batch mode
print("\n6. Comparison with Pure Fed-Batch (optional benchmark):")
print("  Testing pure fed-batch for comparison...")
sim_fb = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    operating_mode=OperatingMode.FEDBATCH,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=True
)
result_fb = sim_fb.run_simulation()
batch_result = sim_fb.data.batches[0]
print(f"    Fed-batch: Xv_final={batch_result.Xv_final:.2e} cells/L, t_end={batch_result.t_end:.1f} h")
print(f"    Conc. Fed-batch: Xv_final={batch_1.Xv_final:.2e} cells/L, t_end={batch_1.t_end:.1f} h")
print(f"    Cell density improvement: {(batch_1.Xv_final/batch_result.Xv_final - 1)*100:.1f}%")
print("\n✓ All concentrated fed-batch-specific validations completed")

Concentrated Fed-Batch-Specific Validations:

1. Mode Transition Detection (Batch 1):
  Mode transition detected at t=144.55 h
  Conditions at transition:
    Xv: 8.30e+09 cells/L
    Volume: 2000.00 L
    Lactate: 4.81 mmol/L
    Ammonia: 2.77 mmol/L
    Dilution rate: 0.0498 1/h
 Termination reason: <animakinetica.models.s.perfusionharveststopevent object at 0x0000020fd4deda90>

2. Volume Profile:
  V_initial: 600.00 L
  V_max: 2000.00 L
  V_final: 2000.00 L

  Fed-batch phase (0 to 144.6 h):
    Volume increase: 1400.00 L
  Perfusion phase (144.6 to 216.5 h):
    Volume CV: 0.00%

3. Dilution Rate Profile:
  D_initial: 0.0000 1/h
  D_max: 0.2000 1/h
  D_final: 0.2000 1/h
  D_mean: 0.0559 1/h

4. Cell Density Profile:
  Batch 1:
    Xv_initial: 6.00e+08 cells/L
    Xv_max: 5.00e+10 cells/L (fold: 83.33x)
    Xv_final: 5.00e+10 cells/L
  Batch 2:
    Xv_initial: 6.00e+08 cells/L
    Xv_max: 5.00e+10 cells/L (fold: 83.33x)
    Xv_final: 5.00e+10 cells/L
  Batch 3:
    Xv_initial: 6.00e

ValueError: operands could not be broadcast together with shapes (0,) (2,) 

In [ ]:
sim.data.plot_biomass_metabolites()

In [ ]:
sim.data.plot_volume_feed()

In [ ]:
sim.data.plot_pH_control(batch_number=1)

In [ ]:
sim.data.plot_gas_transfer()

In [ ]:
sim.data.plot_kinetics()

## 8. Feed rates and glucose concentration visualizations

In [7]:
# Comparison plot: perfusion profiles and glucose concentration across all perfusion strategies
import copy
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from animakinetica.analysis.batch_visualization import (
    plot_batch_volume_feed,
    plot_batch_biomass_metabolites,
)

# Use same simulation setup for fair comparison
perfusion_strategies_compare = [
    PerfusionStrategy.CONSTANT,
    PerfusionStrategy.TOXICITY_BASED_STEP,
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL,
    PerfusionStrategy.CSPR,
    PerfusionStrategy.GLUCOSE_SETPOINT,
]

color_map = {
    PerfusionStrategy.CONSTANT: "#1f77b4",
    PerfusionStrategy.TOXICITY_BASED_STEP: "#ff7f0e",
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL: "#2ca02c",
    PerfusionStrategy.CSPR: "#d62728",
    PerfusionStrategy.GLUCOSE_SETPOINT: "#9467bd",
}

fig_compare = make_subplots(
    rows=1, cols=6,
    subplot_titles=(
        "Process Perfusion Rate Comparison",
        "Base Feed Rate Comparison",
        "Total Inlet Rate Comparison",
        "Volume Comparison",
        "Glucose Comparison",
        "Cell Density Comparison"
    ),
    horizontal_spacing=0.08
)

params_compare = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch", "perfusion"])

for ps in perfusion_strategies_compare:
    params_test = copy.deepcopy(params_compare)
    params_test.update("excess_kinetics_batch")

    sim_test = CellCultureSimulation(
        parameters=params_test,
        kinetics=Kinetics.EXCESS,
        feed_strategy=FeedStrategy.CONSTANT,
        operating_mode=OperatingMode.CONCENTRATED_FEDBATCH,
        perfusion_strategy=ps,
        gas_controller=GasController.CASCADE_N_FG_XO2,
        performance_mode=False,
        solver_method="BDF",
    )

    res = sim_test.run_simulation(number_of_batches=1)

    # Reuse existing plotting utilities and extract traces from their outputs
    fig_volume = plot_batch_volume_feed(res, batch_number=1, layout_kwargs={"showlegend": False})
    fig_biomass = plot_batch_biomass_metabolites(res, batch_number=1, layout_kwargs={"showlegend": False})

    process_rate_trace = next(tr for tr in fig_volume.data if tr.name == "Process Feed Rate")
    base_rate_trace = next(tr for tr in fig_volume.data if tr.name == "Base Feed Rate")
    total_rate_trace = next(tr for tr in fig_volume.data if tr.name == "Total Inlet Rate")
    volume_trace = next(tr for tr in fig_volume.data if tr.name == "Volume")
    glucose_trace = next(tr for tr in fig_biomass.data if tr.name == "Glucose")
    biomass_trace = next(tr for tr in fig_biomass.data if tr.name == "Cell Density")

    label = ps.name
    color = color_map.get(ps, None)

    fig_compare.add_trace(
        go.Scatter(
            x=process_rate_trace.x,
            y=process_rate_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=True,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_process: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=1,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=base_rate_trace.x,
            y=base_rate_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_base: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=2,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=total_rate_trace.x,
            y=total_rate_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_total: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=3,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=volume_trace.x,
            y=volume_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color, dash="dot"),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>V: %{y:.2f} L<extra></extra>",
        ),
        row=1, col=4,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=glucose_trace.x,
            y=glucose_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>Glucose: %{y:.2f} mmol/L<extra></extra>",
        ),
        row=1, col=5,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=biomass_trace.x,
            y=biomass_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>Cell Density: %{y:.2f} 10⁶ cells/mL<extra></extra>",
        ),
        row=1, col=6,
    )

for col in range(1, 6):
    fig_compare.update_xaxes(title_text="Time (h)", row=1, col=col)

fig_compare.update_yaxes(title_text="Process Perfusion Rate (L/h)", row=1, col=1)
fig_compare.update_yaxes(title_text="Base Feed Rate (L/h)", row=1, col=2)
fig_compare.update_yaxes(title_text="Total Inlet Rate (L/h)", row=1, col=3)
fig_compare.update_yaxes(title_text="Volume (L)", row=1, col=4)
fig_compare.update_yaxes(title_text="Glucose (mmol/L)", row=1, col=5)
fig_compare.update_yaxes(title_text="Cell Density (10⁶ cells/mL)", row=1, col=6)

fig_compare.update_layout(
    title_text="Perfusion Strategy Comparison",
    template="plotly_white",
    height=500,
    width=3200,
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
)

fig_compare.show()

Loading parameters from 6 file(s) sequentially...
  [1/6] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
  [6/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\animakinetica\config\pa

## Summary

All tests completed successfully for **CONCENTRATED_FEDBATCH** operating mode:
- ✓ Simulation setup and execution
- ✓ Single and multi-batch runs
- ✓ Data integrity validation
- ✓ Performance metrics
- ✓ Concentrated fed-batch-specific validations (mode transition, volume profile, combined feeding/perfusion)
- ✓ Comparison with pure fed-batch mode